# Música urbana chilena: aprender a analizar redes con R
## Clase inicial de 80 minutos · Google Colab

Usaremos los datos reales de **mu-ch**, dicotomizados, para comparar dos periodos. Cada sección sigue el mismo orden: **pregunta, explicación breve, función específica de R, cálculo e interpretación**. Después aplicaremos el mismo procedimiento a otro periodo o atributo.

Los cálculos se realizan cuando llegamos a cada tema. Al comienzo solo cargamos los paquetes, leemos el Excel y creamos las redes. Las funciones auxiliares de dibujo conservan el estilo visual de mu-ch; no sustituyen las funciones de análisis que aprenderemos a aplicar.

Después de cada ejercicio hay una tabla **P1 / P2** y preguntas para interpretar. Las tablas tienen una salida de referencia verificada para poder verlas al abrir el cuaderno; al ejecutar sus celdas vuelven a calcularse con los datos cargados. Intenta el ejercicio antes de mirar la comprobación.

### Abrir en Colab
1. Sube este `.ipynb` a [Google Colab](https://colab.research.google.com/) y comprueba que el lenguaje del entorno sea **R**.
2. Sube `base_mu_ch_binaria.xlsx` y `ego_pablo_ficticio.xlsx` al panel **Archivos**.
3. Ejecuta las celdas en orden con su botón o Mayús+Intro. Guarda una copia del cuaderno en Drive.
4. Si reinicias el entorno, vuelve a subir los Excel y ejecuta desde el comienzo.

**Preparación docente:** instalar los paquetes antes de la clase. La primera instalación queda fuera de los 80 minutos. Todos los ejercicios se hacen en la red completa real. El ejemplo egocentrado es solo visual y ficticio.

| Minutos | Actividad |
|---|---|
| 0–10 | Abrir el Excel y reconocer el tipo y formato de los datos |
| 10–18 | Qué es una función; leer tablas y crear las redes |
| 18–28 | Comparar visualmente redes completas y egocentradas |
| 28–38 | Aplicar medidas de red completa |
| 38–56 | Calcular grado, intermediación y eigenvector; ordenar artistas |
| 56–68 | Calcular homofilia por país y generación |
| 68–78 | Aplicar Louvain y Leiden |
| 78–80 | Cierre |

No es necesario memorizar las instrucciones ni terminar todas las extensiones. Los ejercicios principales repiten pocas líneas. Las soluciones quedan al final.


## 1. Primero mirar la base: ¿qué tipo de datos tenemos? (0–10 min)

Abre **base_mu_ch_binaria.xlsx** antes de calcular. Una base de SNA permite saber **quiénes son los actores y cómo se relacionan**. Una tabla con edades o países, por sí sola, no registra la red.

Conviene separar dos preguntas: **qué clase de red estudiamos** y **cómo la guardamos**.

### A. Qué clase de red estudiamos
| Distinción | Qué significa | Nuestra base real |
|---|---|---|
| Completa / egocentrada | Universo delimitado de actores y vínculos / entorno de un actor seleccionado | **Completa**, dentro de la frontera final de mu-ch |
| Un modo / dos modos | Un tipo de entidad / conexiones entre dos conjuntos distintos | **Un modo:** artistas, tanto solistas como agrupaciones |
| No dirigida / dirigida | Relación conjunta / relación con emisor y receptor | **No dirigida:** A colabora con B implica la misma pareja que B con A |
| Binaria / valorada | Presencia o ausencia / cantidad o intensidad | **Binaria:** una pareja con una o diez pistas vale 1 |
| Una observación / varias ventanas | Una fotografía / comparación de momentos | **Dos ventanas independientes**, no acumulativas |

Solistas y agrupaciones son categorías del atributo `tipo` dentro de la misma clase de entidad, artistas. No convierten esta base en una red de dos modos. Una red artista–evento sí conectaría dos conjuntos de entidades.

**Completa** no significa que todos colaboren con todos ni que tengamos toda la música urbana. Significa que analizamos los nodos y vínculos del universo delimitado por la investigación. El filtro de mu-ch exige que cada vínculo tenga al menos un extremo del núcleo validado; esa cobertura afecta lo que podemos observar.

### B. Cómo podemos guardar una red
| Formato | Qué representa cada fila | Ejemplo |
|---|---|---|
| **Lista de vínculos** | Una pareja conectada | Artista A, artista B, periodo P1 |
| **Matriz de adyacencia** | Un actor; las columnas son los mismos actores | La celda A–B contiene 1 si colaboran y 0 si no |
| **Matriz de afiliación o incidencia de dos modos** | Un actor; las columnas son eventos, canciones u otras entidades | La celda artista–canción indica participación |
| **Tabla de nodos y atributos** | Un actor, con sus características | ID, nombre, país, generación; necesita vínculos para representar la red |

**Nuestro Excel combina una lista de vínculos con una tabla de nodos y atributos.** Ambas se conectan mediante el ID y el periodo. Convertiremos esa lista a un objeto de red de igraph y luego miraremos su matriz de adyacencia. Son representaciones del mismo patrón de conexiones, no redes distintas.

### C. Las hojas del Excel
| Hoja | Unidad de cada fila | Qué contiene |
|---|---|---|
| **nodos** | Un artista en un periodo | `id`, `artista`, `periodo`, `pais`, `generacion`, `tipo`, `genero`, `sello` |
| **relaciones** | Una pareja de artistas en un periodo | `origen`, `destino`, `periodo`, `vinculo` |
| **periodos** | Una ventana de observación | Fechas de comienzo y final |
| **diccionario** | Una explicación de variable o decisión | Significado de columnas, frontera y procedencia |

Un mismo artista puede aparecer dos veces en nodos porque participa en ambos periodos. Su ID es el mismo. Una misma pareja puede aparecer una vez por periodo; dentro de cada ventana no hay duplicados ni vínculos de un artista consigo mismo.

La lista está agregada por **pareja y periodo**: no es una lista de canciones ni un registro de eventos individuales. El número de artistas cambia entre ventanas; no es un panel de los mismos nodos. Un artista ausente de la tabla de nodos de P1 no se añade como aislado de P1.

En relaciones, `vinculo` siempre vale **1**: la hoja solo enumera parejas con colaboración. Los **ceros están implícitos** en las parejas ausentes dentro del universo de cada periodo. No incluimos filas con cero. Los nombres `origen` y `destino` identifican extremos; no indican dirección.

**Mini ejemplo inventado:** si A–B y B–C están en la lista, pero A–C no, la matriz tiene ceros en A–C y C–A. Como no hay dirección, A–B y B–A muestran el mismo valor.

**Mirar antes de ejecutar:** busca Pablo Chill-E en nodos. Identifica un atributo. Luego observa cómo dos IDs forman una fila en relaciones. ¿Qué cambiaría si guardáramos artistas y canciones en lugar de parejas de artistas?


## Mapa inicial: ¿qué significa que un artista sea central?

Una centralidad describe una **posición en las relaciones observadas**. Cada medida responde una pregunta distinta; no es una nota general de talento o fama.

| Medida | Pregunta sencilla | Ejemplo musical | ¿La calculamos? |
|---|---|---|---|
| **Grado** | ¿Con cuántas personas conecta directamente? | Tiene colaboraciones con 15 artistas distintos | Sí |
| **Intermediación / betweenness** | ¿Cuántos caminos cortos entre otras personas pasan por él? | Conecta dos grupos de artistas que, sin ese puente, quedan más lejos o separados | Sí |
| **Eigenvector** | ¿Sus contactos también están bien conectados? | Colabora dentro de un sector donde sus contactos tienen posiciones centrales | Sí, en el componente mayor |
| **Cercanía / closeness** | ¿Cuántos pasos necesita para alcanzar al resto? | Está a pocos vínculos de los demás artistas del componente | Solo como concepto |

**Grado y fuerza:** grado cuenta colaboradores distintos. Fuerza sumaría repeticiones, pero aquí todos los vínculos valen 1, así que coincide con grado.

### Fórmulas de referencia: primero comprender la pregunta
No hay que resolverlas a mano. El código las calcula. En estas expresiones, $a_{ij}=1$ si existe vínculo entre $i$ y $j$, y 0 si no. $n$ es el número de nodos de la red utilizada.

**Grado:** sumar las conexiones de un nodo.

$$k_i=\sum_j a_{ij}$$

**Intermediación:** para cada pareja de otros nodos, sumar la parte de sus caminos mínimos que pasa por $i$.

$$B_i=\sum_{s<t;\;s,t\ne i}\frac{\sigma_{st}(i)}{\sigma_{st}},\qquad B_i^{\mathrm{norm}}=\frac{B_i}{(n-1)(n-2)/2}$$

$\sigma_{st}$ cuenta todos los caminos mínimos entre $s$ y $t$; $\sigma_{st}(i)$ cuenta los que pasan por $i$. Las parejas sin camino no aportan. La fórmula normalizada es la que usamos en la red no dirigida: divide por todas las parejas posibles entre los otros nodos.

**Eigenvector:** la puntuación de $i$ es proporcional a la suma de puntuaciones de sus vecinos.

$$x_i=\frac{1}{\lambda}\sum_j a_{ij}x_j$$

$\lambda$ es la constante que permite resolver las puntuaciones conjuntamente. Después se escalan para que el máximo valga 1. Se calcula dentro del componente mayor.

**Cercanía:** inversa de la distancia media al resto, dentro de un componente conectado de $n_c$ nodos.

$$C_i=\frac{n_c-1}{\sum_{j\ne i}d(i,j)}$$

$d(i,j)$ cuenta los vínculos del camino más corto entre ambos. Esta última medida se muestra para comparar conceptos; no agregamos un ejercicio de cercanía.


## 2. Qué es una función y cómo la usamos (10–18 min)

Una **función** es una instrucción preparada que realiza una tarea. Le entregamos algo y devuelve un resultado. Por ejemplo, `vcount(red1)` recibe la red del primer periodo y devuelve cuántos nodos tiene.

```r
nombre_funcion(dato, opcion = valor)
```

- **Nombre:** indica qué tarea pedimos. `head()` muestra las primeras filas; `degree()` calcula grado.
- **Paréntesis:** contienen los datos y opciones, llamados **argumentos**.
- **Coma:** separa argumentos. En `head(nodos, 6)`, pedimos seis filas de nodos.
- **`<-`:** guarda un resultado con un nombre. `cantidad <- vcount(red1)` guarda el conteo; escribir `cantidad` después lo muestra. El nombre lo elegimos nosotros.
- **Comillas:** encierran texto. `"P1"` es una etiqueta; `red1` sin comillas es un objeto ya creado.
- **`$`:** accede a una columna o parte de un resultado. `nodos$artista` recupera nombres; `resultado$vector` recupera el vector calculado por una función.
- **`[]`:** selecciona elementos. En una tabla, antes de la coma elegimos filas y después elegimos columnas.

Un **paquete** reúne funciones. `igraph` permite analizar redes; `readxl` lee Excel. `library(igraph)` carga sus funciones. En cambio, funciones como `head()`, `mean()` y `order()` ya vienen con R.

Primero ejecutaremos las instrucciones tal como están. Luego cambiaremos un dato o argumento. No escribiremos funciones nuevas para calcular las medidas.


In [ ]:
# Instalar solo los paquetes que faltan. Preparar antes de comenzar la clase.
paquetes <- c("igraph", "readxl", "ggraph", "ggplot2")
# Comprobamos cuáles de esos paquetes faltan.
faltan <- paquetes[!vapply(paquetes, requireNamespace, logical(1), quietly = TRUE)]
# Instalamos únicamente los paquetes que faltan.
if (length(faltan)) install.packages(faltan, repos = "https://cloud.r-project.org")
# Cargamos los paquetes sin mostrar sus mensajes de inicio.
suppressPackageStartupMessages({library(igraph); library(readxl); library(ggraph); library(ggplot2)})
# Ajustamos el tamaño de los dibujos y el ancho del texto.
options(repr.plot.width = 12, repr.plot.height = 9, width = 100)


### Leer y mirar las tablas
`read_excel()` recibe el archivo y la hoja. `as.data.frame()` convierte lo leído a una tabla de R. La operación de adentro se realiza primero. En estas dos líneas guardamos las tablas con los nombres `nodos` y `relaciones`.


In [ ]:
# Leemos la hoja del Excel y la guardamos como tabla.
nodos <- as.data.frame(read_excel("base_mu_ch_binaria.xlsx", sheet = "nodos"))
# Leemos la hoja del Excel y la guardamos como tabla.
relaciones <- as.data.frame(read_excel("base_mu_ch_binaria.xlsx", sheet = "relaciones"))
# Mostramos las primeras filas de la tabla.
head(nodos, 6)
# Mostramos las primeras filas de la tabla.
head(relaciones, 6)


### Elegir un periodo y crear el objeto de red
`subset()` selecciona filas que cumplen una condición. `==` pregunta si dos valores son iguales. Guardaremos los artistas y relaciones de cada ventana por separado.

`graph_from_data_frame()` recibe la lista de vínculos. **Sus dos primeras columnas deben ser los IDs de la pareja**. Por eso seleccionamos `origen` y `destino`. `vertices` incorpora la tabla de nodos y sus atributos; `directed = FALSE` indica que no hay dirección. No pasamos pesos: las redes quedan binarias.


In [ ]:
# Seleccionamos las filas del periodo indicado.
nodos1 <- subset(nodos, periodo == "P1")
# Seleccionamos las filas del periodo indicado.
pares1 <- subset(relaciones, periodo == "P1", select = c(origen, destino))
# Creamos la red sin dirección a partir de nodos y parejas.
red1 <- graph_from_data_frame(pares1, directed = FALSE, vertices = nodos1)

# Seleccionamos las filas del periodo indicado.
nodos2 <- subset(nodos, periodo == "P2")
# Seleccionamos las filas del periodo indicado.
pares2 <- subset(relaciones, periodo == "P2", select = c(origen, destino))
# Creamos la red sin dirección a partir de nodos y parejas.
red2 <- graph_from_data_frame(pares2, directed = FALSE, vertices = nodos2)


### La misma red, guardada como matriz
`as_adjacency_matrix()` convierte el grafo a una matriz. `sparse = FALSE` pide verla como una tabla convencional. Mostramos solo seis filas y seis columnas: el resto continúa en el objeto.

La matriz es cuadrada y simétrica. La diagonal tiene cero: no registramos colaboraciones de un artista consigo mismo. Las dos celdas simétricas de una pareja representan **un solo vínculo**.


In [ ]:
# Convertimos la red a una matriz de ceros y unos.
matriz1 <- as_adjacency_matrix(red1, sparse = FALSE)
# Miramos solo una esquina de la matriz.
matriz1[1:6, 1:6]


## 3. Comparar visualmente dos tipos de red (18–28 min)

La celda de abajo prepara **solo los gráficos**. Define `ver_red()` y `ver_ego()`, funciones de dibujo creadas para esta clase. El cálculo de cada medida y algoritmo vendrá después, con las funciones específicas de igraph a la vista.

Los gráficos usan el estilo de mu-ch, posiciones comunes y escalas iguales entre periodos. Para asignar tamaño a los puntos, el dibujo cuenta sus contactos, pero no prepara los rankings ni las tablas de análisis. Las coordenadas se calculan con la unión de periodos solamente para la visualización.


In [ ]:
# Datos inventados del ejemplo egocentrado. Solo para las figuras.
ego_nodos <- as.data.frame(read_excel("ego_pablo_ficticio.xlsx", sheet = "nodos"))
# Leemos la hoja del Excel y la guardamos como tabla.
ego_relaciones <- as.data.frame(read_excel("ego_pablo_ficticio.xlsx", sheet = "relaciones"))


In [ ]:
# Preparación gráfica docente. Se puede plegar esta celda.
# Estilo mu-ch: graphopt, bordes suaves, puntos con borde y leyenda inferior.
# Las coordenadas se calculan UNA vez con la union de los dos periodos, solo para visualizar.
pares_union <- unique(relaciones[c("origen", "destino")])
ids_union <- sort(unique(nodos$id))
red_union <- graph_from_data_frame(pares_union, directed = FALSE, vertices = data.frame(id = ids_union))
set.seed(123)
coordenadas <- layout_with_graphopt(red_union, niter = 1000, charge = .02,
                                    spring.length = 65, spring.constant = .6)
rownames(coordenadas) <- V(red_union)$name
lim_x <- range(coordenadas[, 1])
lim_y <- range(coordenadas[, 2])
tamano_maximo <- max(degree(red1), degree(red2))

tema_red <- theme_void(base_size = 12) +
  theme(plot.title = element_text(face = "bold", hjust = .5),
        plot.subtitle = element_text(hjust = .5),
        legend.position = "bottom", plot.margin = margin(12, 12, 12, 12))

ver_red <- function(red, titulo = NULL, comunidad = NULL) {
  if (is.null(titulo)) titulo <- if (vcount(red) == 213) "P1: 2014-febrero 2022" else "P2: marzo 2022-2024"
  grafo <- red
  V(grafo)$grado <- degree(grafo)
  destacados <- order(-V(grafo)$grado, V(grafo)$name)[1:3]
  V(grafo)$etiqueta <- ""
  V(grafo)$etiqueta[destacados] <- V(grafo)$artista[destacados]
  if (is.null(comunidad)) {
    V(grafo)$color_grupo <- V(grafo)$pais
    titulo_leyenda <- "Origen"
  } else {
    V(grafo)$color_grupo <- factor(as.character(membership(comunidad)),
                                   levels = as.character(sort(unique(membership(comunidad)))))
    titulo_leyenda <- "Comunidad"
  }
  xy <- coordenadas[V(grafo)$name, , drop = FALSE]
  dibujo <- ggraph(grafo, layout = "manual", x = xy[, 1], y = xy[, 2]) +
    geom_edge_link(colour = "grey25", alpha = .18, edge_width = .28) +
    geom_node_point(aes(size = grado, fill = color_grupo, shape = tipo),
                    colour = "grey15", stroke = .3, alpha = .85) +
    geom_node_text(aes(label = etiqueta), repel = TRUE, size = 3, max.overlaps = Inf,
                   box.padding = .8, point.padding = .6, force = 5, max.iter = 5000, seed = 123) +
    scale_shape_manual(values = c(Solista = 21, Agrupacion = 24), drop = FALSE) +
    scale_size_continuous(range = c(1, 4.5), limits = c(0, tamano_maximo), guide = "none") +
    coord_equal(xlim = lim_x, ylim = lim_y, clip = "off") +
    labs(title = titulo, subtitle = "Vinculos binarios. Tamano del nodo: colaboradores distintos.",
         fill = titulo_leyenda, shape = "Tipo") + tema_red +
    guides(fill = guide_legend(nrow = 3, byrow = TRUE, override.aes = list(size = 4, shape = 21, alpha = 1)),
           shape = guide_legend(override.aes = list(size = 4, fill = "grey80", alpha = 1)))
  if (is.null(comunidad)) dibujo + scale_fill_manual(values = c(Chile = "#1B9E77", Extranjero = "#D95F02"), drop = FALSE)
  else dibujo + scale_fill_viridis_d(option = "turbo", begin = .03, end = .93)
}


# Ejemplo separado: solo visual, totalmente ficticio.
ver_ego <- function(periodo = "P1") {
  actores <- ego_nodos[ego_nodos$periodo == periodo, ]
  pares <- ego_relaciones[ego_relaciones$periodo == periodo, c("origen", "destino")]
  red <- graph_from_data_frame(pares, directed = FALSE, vertices = actores)
  ids <- c("EGO", "R1", "R2", "R3", "B1", "B2", "V1", "M1", "D1", "S1")
  angulo <- seq(0, 2*pi, length.out = 10)[1:9]
  xy <- rbind(c(0, 0), cbind(cos(angulo), sin(angulo)))
  rownames(xy) <- ids
  xy <- xy[V(red)$name, , drop = FALSE]
  V(red)$grupo <- ifelse(V(red)$rol %in% c("Rapero", "Rapera"), "Rap", V(red)$rol)
  V(red)$etiqueta <- ifelse(V(red)$name == "EGO", "Pablo Chill-E\nEgo del ejemplo",
                          paste(V(red)$artista, V(red)$rol, sep = "\n"))
  V(red)$tamano <- ifelse(V(red)$name == "EGO", 10, 6)
  ggraph(red, layout = "manual", x = xy[, 1], y = xy[, 2]) +
    geom_edge_link(colour = "grey50", alpha = .55, edge_width = .6) +
    geom_node_point(aes(fill = grupo, size = tamano), shape = 21, colour = "grey20") +
    geom_node_text(aes(label = etiqueta), repel = TRUE, size = 3.5, max.overlaps = Inf,
                   point.padding = 1, box.padding = .5, force = 3, seed = 123) +
    scale_fill_manual(values = c(Rap = "#3B6AB7", Beatmaker = "#D99522", Audiovisual = "#9A64B6", Management = "#D05B62", DJ = "#269E9B", Sonido = "#6A8D35")) +
    scale_size_identity() + coord_equal(xlim = c(-1.4, 1.4), ylim = c(-1.3, 1.3), clip = "off") +
    labs(title = paste("Red egocentrada ficticia:", periodo),
         subtitle = "TODOS los vinculos y contactos son inventados. No representa su equipo real.",
         fill = "Funcion en el ejemplo") + tema_red
}


### Red completa real
**P1:** enero de 2014 a febrero de 2022. **P2:** marzo de 2022 a diciembre de 2024. Una línea indica al menos una pista acreditada compartida; no demuestra amistad. Color: Chile o Extranjero. Forma: solista o agrupación. Tamaño: cantidad de colaboradores. Solo se etiquetan tres artistas para mantener legibilidad.


In [ ]:
# Dibujamos este grafo para observar su estructura.
ver_red(red1)


**Lectura preliminar de la red completa P1:** predominan los nodos chilenos; se observan contactos extranjeros y pequeños conjuntos separados. Pablo Chill-E, Stailok y El BAI aparecen destacados por sus colaboradores. El dibujo sugiere diferencias de posición, pero contar componentes y comparar medidas permitirá comprobarlas. La posición en el papel no es geográfica.


In [ ]:
# Dibujamos este grafo para observar su estructura.
ver_red(red2)


**Lectura preliminar de la red completa P2:** se ve una red más grande, con más contactos extranjeros y un conjunto central amplio. El BAI y Jairo Vera aparecen con muchos colaboradores. Hay más puntos y líneas, pero eso no demuestra mayor densidad: debemos dividir los vínculos por todas las parejas posibles. El dibujo no demuestra por sí solo que desaparezcan desigualdades.


### Red egocentrada ficticia de Pablo Chill-E
Mostramos ego, sus contactos directos y relaciones entre esos contactos. **Todos los contactos y vínculos son inventados**; no describen su equipo real ni se extraen de mu-ch. Los personajes Rapero A, Beatmaker A, Manager A, etc. son ficticios. Aquí la relación es trabajo conjunto en proyectos musicales imaginarios y P1/P2 son momentos sin fechas históricas.

`ego_pablo_ficticio.xlsx` también usa hojas de nodos y relaciones. **El formato de archivo es el mismo; cambia el alcance de la red:** seleccionamos el entorno de ego, en lugar del universo completo de mu-ch.

Que ego esté conectado con todos depende de cómo construimos este recorte. Observa también las líneas entre sus contactos. Este ejemplo termina aquí: **todos los ejercicios siguientes usan la red completa real**.


In [ ]:
# Dibujamos este grafo para observar su estructura.
ver_ego("P1")


**Lectura preliminar del ejemplo ficticio P1:** ego tiene cinco contactos directos. Los contactos también trabajan entre sí: por ejemplo, Rapero A con Beatmaker A y Realizador A, y Rapera B con Manager A. Los colores distinguen funciones. El equipo no es una estrella de vínculos exclusivamente hacia ego. Todo es inventado.


In [ ]:
# Dibujamos este grafo para observar su estructura.
ver_ego("P2")


**Lectura preliminar del ejemplo ficticio P2:** aparecen nueve contactos directos y más vínculos entre contactos, incluidos Beatmaker A–Beatmaker B y DJ A–Sonidista A. Los colores permiten ver cooperación entre funciones distintas. El nodo de ego es más grande para identificarlo; este recorte no prueba mayor prestigio ni describe el equipo real de Pablo Chill-E.


## 4. Medidas de la red completa (28–38 min)

**Pregunta:** ¿cuántos artistas participan, cuánto colaboran y cuán conectados están?

Aplicaremos una función por pregunta. Densidad expresa la proporción de parejas posibles que tienen vínculo; 0,02 equivale aproximadamente a dos de cada cien parejas posibles. Grado medio indica colaboradores por artista, en promedio.


In [ ]:
vcount(red1)                         # Cantidad de artistas
ecount(red1)                         # Cantidad de parejas colaboradoras
edge_density(red1, loops = FALSE)     # Densidad; no contamos lazos consigo mismo
mean(degree(red1))                   # Promedio de colaboradores


### Ejercicio 1 · aplicar las mismas funciones a P2 (2 min)
Cambia **red1 por red2** en las cuatro líneas. ¿Hay más artistas y más parejas? ¿La densidad sube también? La densidad puede bajar si crecen más rápido las parejas posibles. Las ventanas tienen distinta duración: los conteos no son tasas anuales comparables.


In [ ]:
# Contamos los artistas de la red.
vcount(red1)
# Contamos las parejas que colaboran.
ecount(red1)
# Calculamos la proporción de parejas posibles que colaboran.
edge_density(red1, loops = FALSE)
# Calculamos el promedio de colaboradores por artista.
mean(degree(red1))


### Comparar después del ejercicio 1
Ejecuta esta comprobación **después de intentar P2**. La tabla reúne los resultados de ambos periodos, calculados ahora con las mismas funciones. `data.frame()` organiza las columnas y `round()` reduce los decimales para leer mejor.


In [ ]:
# Armamos una fila por medida y una columna por periodo.
comparacion_red <- data.frame(
  medida = c("Artistas", "Parejas colaboradoras", "Densidad", "Colaboradores promedio"),
  P1 = c(vcount(red1), ecount(red1), edge_density(red1, loops = FALSE), mean(degree(red1))),
  P2 = c(vcount(red2), ecount(red2), edge_density(red2, loops = FALSE), mean(degree(red2))))
# Mostramos artistas y parejas sin decimales; densidad con cuatro y promedio con dos.
comparacion_red$P1 <- sprintf(c("%.0f", "%.0f", "%.4f", "%.2f"), comparacion_red$P1)
comparacion_red$P2 <- sprintf(c("%.0f", "%.0f", "%.4f", "%.2f"), comparacion_red$P2)
comparacion_red


medida,P1,P2
Artistas,213,564
Parejas colaboradoras,410,1579
Densidad,0.0182,0.0099
Colaboradores promedio,3.85,5.60


### Por qué baja la densidad: coincide con la tesis

La densidad compara **parejas que colaboran / todas las parejas que podrían colaborar**. En una red no dirigida y sin lazos, las parejas posibles son $n(n-1)/2$:

| Cantidad | P1 | P2 |
|---|---:|---:|
| Artistas | 213 | 564 |
| Parejas con colaboración | 410 | 1579 |
| Parejas posibles | 22578 | 158766 |
| Densidad, aproximadamente | 1,82 % | 0,99 % |

Las colaboraciones aumentan unas **3,85 veces**, pero las parejas posibles aumentan unas **7,03 veces**. Por eso la fracción baja. No disminuye la cantidad de colaboraciones registradas; disminuye la proporción de parejas posibles que colabora.

**Esto coincide con los resultados de la tesis**, donde la red crece y el porcentaje dentro del componente mayor aumenta, mientras la densidad baja. El escenario ficticio anterior de 120 nodos fijos sí tenía aumento de densidad; ya no es la base de estos ejercicios.

También importa el tiempo: las ventanas originales tienen distinta duración. Los totales son descriptivos y no deben leerse como tasas anuales directamente comparables.


**Para concluir:** ¿aumentan los artistas y las parejas? ¿Por qué baja la densidad mientras sube el promedio de colaboradores? El número de parejas posibles también cambia. Las ventanas tienen distinta duración: no interpretes los conteos como tasas anuales.


### Componentes e integración
`components()` devuelve varias piezas de información. `$no` indica cuántos componentes hay y `$csize` sus tamaños. Un componente reúne actores conectados por caminos, aunque no todos colaboren directamente. `max()` busca el tamaño mayor; al dividirlo por el total y multiplicar por 100 obtenemos el porcentaje dentro de ese componente.


In [ ]:
# Buscamos los grupos conectados y guardamos sus tamaños.
componentes1 <- components(red1)
# Mostramos la cantidad de componentes.
componentes1$no
# Mostramos cuántos artistas tiene el componente mayor.
max(componentes1$csize)
# Calculamos qué porcentaje de artistas está en el componente mayor.
100 * max(componentes1$csize) / vcount(red1)
sum(degree(red1) == 0)                # Cuenta artistas con cero colaboradores


**Centralización:** describe cuánto se concentran los contactos de toda la red en una persona. `centr_degree()` devuelve una lista; `$centralization` selecciona la medida. Se muestra entre 0 y 1. No es el porcentaje de artistas centrales.


In [ ]:
# Calculamos cuánto se concentran los contactos en la red.
centr_degree(red1, loops = FALSE, normalized = TRUE)$centralization


### Repetición guiada: componentes y centralización en P2
El docente acaba de aplicar las funciones a P1. Repite el bloque de componentes y `centr_degree()` usando red2. Después ejecuta esta tabla de comprobación.


In [ ]:
# Completa cada ____ con red2.
# Luego quita el # inicial SOLO de las líneas de código para ejecutarlas.
# Así la plantilla sin completar no produce errores al ejecutar todo el cuaderno.
# componentes2 <- components(____)
# componentes2$no
# max(componentes2$csize)
# 100 * max(componentes2$csize) / vcount(____)
# sum(degree(____) == 0)
# centr_degree(____, loops = FALSE, normalized = TRUE)$centralization


In [ ]:
# Calculamos los componentes del segundo periodo en este momento.
componentes2 <- components(red2)
# Juntamos las medidas de integración y concentración.
comparacion_integracion <- data.frame(
  medida = c("Componentes", "Artistas aislados", "Artistas en componente mayor",
             "Porcentaje en componente mayor", "Centralizacion de grado"),
  P1 = c(componentes1$no, sum(degree(red1) == 0), max(componentes1$csize),
         100 * max(componentes1$csize) / vcount(red1),
         centr_degree(red1, loops = FALSE, normalized = TRUE)$centralization),
  P2 = c(componentes2$no, sum(degree(red2) == 0), max(componentes2$csize),
         100 * max(componentes2$csize) / vcount(red2),
         centr_degree(red2, loops = FALSE, normalized = TRUE)$centralization))
# Redondeamos y mostramos la tabla.
comparacion_integracion$P1 <- sprintf(c("%.0f", "%.0f", "%.0f", "%.1f", "%.4f"), comparacion_integracion$P1)
comparacion_integracion$P2 <- sprintf(c("%.0f", "%.0f", "%.0f", "%.1f", "%.4f"), comparacion_integracion$P2)
comparacion_integracion


medida,P1,P2
Componentes,9,8
Artistas aislados,0,0
Artistas en componente mayor,191,545
Porcentaje en componente mayor,89.7,96.6
Centralizacion de grado,0.1102,0.0881


**Para concluir:** ¿qué porcentaje de artistas alcanza el grupo conectado más grande en cada periodo? ¿La mayor integración viene acompañada de mayor concentración de contactos?


## 5. Calcular medidas de nodos y ordenar artistas (38–56 min)

### A. Grado: ¿con cuántos artistas distintos colabora?
`degree()` devuelve un número por nodo, en el orden de la red. Guardamos los valores y los juntamos con sus nombres. `V(red1)$artista` recupera el atributo artista de los vértices; `data.frame()` construye una tabla.


In [ ]:
# Contamos los colaboradores distintos de cada artista.
grado1 <- degree(red1)
# Juntamos los resultados en una tabla.
centralidad1 <- data.frame(artista = V(red1)$artista, grado = as.numeric(grado1))
# Mostramos las primeras filas de la tabla.
head(centralidad1, 6)


`order()` indica cómo ordenar las filas. `decreasing = TRUE` pide valores de mayor a menor. Dentro de `[filas, columnas]`, dejamos vacía la parte de columnas para conservarlas todas. Finalmente, `head()` muestra diez filas.

La línea `rownames(...) <- NULL` vuelve a numerar las filas de la tabla después de ordenarlas. Es solo una ayuda de lectura.

Si dos artistas tienen el mismo valor están empatados: el orden de sus filas no establece una diferencia de centralidad.


In [ ]:
# Ordenamos las filas por la medida, de mayor a menor.
ranking_grado1 <- centralidad1[order(centralidad1$grado, decreasing = TRUE), ]
rownames(ranking_grado1) <- NULL  # Vuelve a numerar las filas para facilitar la lectura
# Mostramos las primeras filas de la tabla.
head(ranking_grado1, 10)


### Ejercicio 2 · calcular y ordenar grado en P2 (3 min)
Toda la sintaxis está preparada. Sustituye **cada `____` por `red2`**. Después quita el `#` inicial de las líneas de código y ejecútalas. Las instrucciones restantes se mantienen. Identifica un artista presente en ambos rankings y observa sus colaboradores.


In [ ]:
# Completa cada ____ con red2.
# Luego quita el # inicial SOLO de las líneas de código para ejecutarlas.
# Así la plantilla sin completar no produce errores al ejecutar todo el cuaderno.
# grado2 <- degree(____)
# centralidad2 <- data.frame(artista = V(____)$artista, grado = as.numeric(grado2))
# ranking_grado2 <- centralidad2[order(centralidad2$grado, decreasing = TRUE), ]
# rownames(ranking_grado2) <- NULL
# head(ranking_grado2, 10)


### Comparar después del ejercicio 2
Esta comprobación calcula y ordena P2 si todavía no completaste el ejercicio. Lee las filas de arriba abajo. Cada fila enfrenta el mismo lugar de la lista en ambos periodos, **no necesariamente al mismo artista**. Valores iguales indican empate.


In [ ]:
# Comprobamos el cálculo y el orden del segundo periodo.
grado2 <- degree(red2)
# Juntamos los resultados en una tabla.
centralidad2 <- data.frame(artista = V(red2)$artista, grado = as.numeric(grado2))
# Ordenamos las filas por la medida, de mayor a menor.
ranking_grado2 <- centralidad2[order(centralidad2$grado, decreasing = TRUE), ]
# Ponemos las diez primeras filas de P1 y P2 una al lado de la otra.
comparacion_grado <- data.frame(
  fila = 1:10,
  artista_P1 = head(ranking_grado1$artista, 10),
  grado_P1 = head(ranking_grado1$grado, 10),
  artista_P2 = head(ranking_grado2$artista, 10),
  grado_P2 = head(ranking_grado2$grado, 10))
# Mostramos la tabla comparativa.
comparacion_grado


fila,artista_P1,grado_P1,artista_P2,grado_P2
1,Pablo Chill-E,27,El BAI,55
2,Stailok,20,Jairo Vera,55
3,El BAI,17,Julianno Sosa,53
4,Maickyel,16,Balbi El Chamako,48
5,Simon la Letra,16,ITHAN NY,45
6,Ana Tijoux,14,Marcianeke,45
7,Andresito Otro Corte,14,Pablo Chill-E,43
8,Balbi El Chamako,14,El Jordan 23,39
9,Carlitos Junior,13,Tunechikidd,36
10,Bayriton,12,Aqua VS,35


**Para concluir:** identifica un artista que aparezca en ambas listas y observa sus colaboradores. ¿Cambia quién ocupa el primer lugar? El orden de artistas empatados no implica diferencias de centralidad.


### B. Intermediación: conectar a otras personas

Imagina que **Ana colabora con X y X colabora con Beto**, pero Ana y Beto no colaboran directamente. Para ir de Ana a Beto por las relaciones registradas, el recorrido más corto es **Ana — X — Beto**. X queda en medio; Ana y Beto son los extremos.

Ahora piensa en dos grupos que colaboran mucho por dentro. Si X es una de las pocas conexiones entre ellos, muchos caminos cortos de un grupo al otro pasan por X. Eso puede darle **alta intermediación aunque no tenga el mayor número de contactos**.

Si hay dos caminos igual de cortos, uno por X y otro por Y, cada uno recibe la mitad de la contribución de esa pareja. Repetimos esta cuenta para todas las parejas de otros artistas. No contamos a X cuando es el punto de partida o llegada.

`betweenness()` realiza esa cuenta. `weights = NA` usa los vínculos como pasos, sin pesos. `normalized = TRUE` divide el resultado por las parejas posibles entre los otros nodos. Por ejemplo, **0,25 equivale al 25 % de ese máximo teórico**, considerando fracciones cuando hay caminos empatados; no significa que controle el 25 % de las canciones o personas.

En esta base describe **caminos de colaboración**, no circulación real de mensajes ni poder comprobado. Una persona puede ser puente entre circuitos sin presentar efectivamente a sus integrantes.

Agregamos la medida a la tabla de artistas y ordenamos del mismo modo que para grado.


In [ ]:
# Calculamos cuánto aparece cada artista como puente en caminos cortos.
centralidad1$intermediacion <- betweenness(red1, directed = FALSE,
                                         weights = NA, normalized = TRUE)
# Ordenamos las filas por la medida, de mayor a menor.
ranking_puentes1 <- centralidad1[order(centralidad1$intermediacion, decreasing = TRUE), ]
rownames(ranking_puentes1) <- NULL  # Vuelve a numerar las filas para facilitar la lectura
# Mostramos las primeras filas de la tabla.
head(ranking_puentes1, 10)


### Repetición guiada: intermediación de P2
Aplica `betweenness()` a red2 y ordena por intermediación, como acaba de hacer el docente con P1. Luego ejecuta la comprobación para ver ambas listas una al lado de la otra.


In [ ]:
# Completa cada ____ con red2.
# Luego quita el # inicial SOLO de las líneas de código para ejecutarlas.
# Así la plantilla sin completar no produce errores al ejecutar todo el cuaderno.
# centralidad2$intermediacion <- betweenness(____, directed = FALSE, weights = NA, normalized = TRUE)
# ranking_puentes2 <- centralidad2[order(centralidad2$intermediacion, decreasing = TRUE), ]
# rownames(ranking_puentes2) <- NULL
# head(ranking_puentes2, 10)


In [ ]:
# Calculamos la intermediación de los artistas de P2.
centralidad2$intermediacion <- betweenness(red2, directed = FALSE,
                                         weights = NA, normalized = TRUE)
# Ordenamos por intermediación, de mayor a menor.
ranking_puentes2 <- centralidad2[order(centralidad2$intermediacion, decreasing = TRUE), ]
# Comparamos las primeras diez filas de los dos rankings.
comparacion_puentes <- data.frame(
  fila = 1:10,
  artista_P1 = head(ranking_puentes1$artista, 10),
  intermediacion_P1 = round(head(ranking_puentes1$intermediacion, 10), 5),
  artista_P2 = head(ranking_puentes2$artista, 10),
  intermediacion_P2 = round(head(ranking_puentes2$intermediacion, 10), 5))
# Mostramos la tabla comparativa.
comparacion_puentes


fila,artista_P1,intermediacion_P1,artista_P2,intermediacion_P2
1,Stailok,0.24752,Pablo Chill-E,0.12799
2,Pablo Chill-E,0.17744,Julianno Sosa,0.1188
3,El Futuro Fuera De Orbita,0.14632,Stailok,0.09422
4,Gino Mella,0.13409,Jairo Vera,0.0809
5,DrefQuila,0.1296,Balbi El Chamako,0.06835
6,Arte Elegante,0.10923,Ak4:20,0.06797
7,Carlitos Junior,0.10864,El BAI,0.06764
8,Movimiento Original,0.103,Marlon Breeze,0.06562
9,ADIONE,0.09748,Vlntna B,0.06111
10,Yiordano Ignacio,0.08209,Lolo OG,0.06001


**Para concluir:** ¿los artistas que encabezan intermediación también encabezan grado? Las columnas comparan lugares de los rankings; no representan necesariamente a la misma persona. Las redes tienen distinto tamaño y la medida está normalizada.


### C. Eigenvector: ¿con quién te conectas?
Eigenvector incorpora la centralidad de los contactos y, a su vez, la de sus contactos. Tener muchos vínculos y tener vínculos con artistas bien posicionados puede producir órdenes diferentes.

Primero seleccionamos el **componente mayor**, porque no queremos mezclar resultados de componentes separados. `which.max()` identifica el componente más grande; `$membership` indica a cuál pertenece cada nodo. `which()` selecciona los nodos que pertenecen a ese componente e `induced_subgraph()` conserva esos nodos y sus vínculos.

Esta selección es un paso preparado: no hace falta memorizarla. **Seguimos trabajando con datos de la red completa, no con una red egocentrada.**


In [ ]:
# Buscamos cuál es el componente más grande.
mayor1 <- which.max(componentes1$csize)
# Conservamos los nodos elegidos y sus vínculos entre sí.
principal1 <- induced_subgraph(red1, which(componentes1$membership == mayor1))
# Contamos los artistas de la red.
vcount(principal1)


Ahora aplicamos **`eigen_centrality()`**. La función devuelve varias piezas; `$vector` contiene los valores de los nodos. El máximo se escala a 1 dentro del componente. Ese valor no es porcentaje de prestigio, y no debe interpretarse como crecimiento absoluto entre periodos.


In [ ]:
# Calculamos eigenvector dentro del componente mayor.
eigen1 <- eigen_centrality(principal1, directed = FALSE, weights = NA)$vector
# Juntamos los resultados en una tabla.
tabla_eigen1 <- data.frame(artista = V(principal1)$artista, eigenvector = as.numeric(eigen1))
# Ordenamos las filas por la medida, de mayor a menor.
ranking_eigen1 <- tabla_eigen1[order(tabla_eigen1$eigenvector, decreasing = TRUE), ]
rownames(ranking_eigen1) <- NULL  # Vuelve a numerar las filas para facilitar la lectura
# Mostramos las primeras filas de la tabla.
head(ranking_eigen1, 10)


### Ejercicio 3 · eigenvector en P2 (3 min)
La sintaxis completa está abajo. Completa **`____` con `red2`** y habilita las líneas quitando su `#` inicial. El bloque selecciona el componente mayor, calcula eigenvector y ordena los artistas. Compara quién encabeza eigenvector y grado en P2.


In [ ]:
# Completa cada ____ con red2.
# Luego quita el # inicial SOLO de las líneas de código para ejecutarlas.
# Así la plantilla sin completar no produce errores al ejecutar todo el cuaderno.
# componentes2 <- components(____)
# mayor2 <- which.max(componentes2$csize)
# principal2 <- induced_subgraph(____, which(componentes2$membership == mayor2))
# eigen2 <- eigen_centrality(principal2, directed = FALSE, weights = NA)$vector
# tabla_eigen2 <- data.frame(artista = V(principal2)$artista, eigenvector = as.numeric(eigen2))
# ranking_eigen2 <- tabla_eigen2[order(tabla_eigen2$eigenvector, decreasing = TRUE), ]
# rownames(ranking_eigen2) <- NULL
# head(ranking_eigen2, 10)


### Comparar después del ejercicio 3
La comprobación aplica `eigen_centrality()` a P2 y reúne los primeros lugares. Recuerda que cada ranking usa **su componente mayor** y que su máximo se escala a 1. Compara posiciones y nombres; un aumento del valor no demuestra un aumento absoluto de prestigio.


In [ ]:
# Preparamos el componente mayor de P2 para comprobar el ejercicio.
componentes2 <- components(red2)
mayor2 <- which.max(componentes2$csize)
principal2 <- induced_subgraph(red2, which(componentes2$membership == mayor2))
# Calculamos eigenvector dentro del componente mayor de P2.
eigen2 <- eigen_centrality(principal2, directed = FALSE, weights = NA)$vector
# Juntamos nombres y valores del segundo periodo.
tabla_eigen2 <- data.frame(artista = V(principal2)$artista, eigenvector = as.numeric(eigen2))
# Ordenamos los artistas de mayor a menor.
ranking_eigen2 <- tabla_eigen2[order(tabla_eigen2$eigenvector, decreasing = TRUE), ]
# Comparamos el mismo lugar de la lista, no necesariamente el mismo artista.
comparacion_eigenvector <- data.frame(
  fila = 1:10,
  artista_P1 = head(ranking_eigen1$artista, 10),
  eigenvector_P1 = round(head(ranking_eigen1$eigenvector, 10), 4),
  artista_P2 = head(ranking_eigen2$artista, 10),
  eigenvector_P2 = round(head(ranking_eigen2$eigenvector, 10), 4))
# Mostramos los diez primeros lugares de ambas listas.
comparacion_eigenvector


fila,artista_P1,eigenvector_P1,artista_P2,eigenvector_P2
1,Balbi El Chamako,1,El BAI,1
2,Drakomafia,0.9458,Jairo Vera,0.932
3,Z Jocker,0.9458,Marcianeke,0.881
4,Bayriton,0.9458,Balbi El Chamako,0.8562
5,Simon la Letra,0.9278,ITHAN NY,0.8317
6,El BAI,0.8961,Tunechikidd,0.7799
7,Maickyel,0.8001,Julianno Sosa,0.75
8,Andresito Otro Corte,0.744,Pablo Chill-E,0.7036
9,King Savagge,0.7188,Jere Klein,0.6889
10,Fabian Riveros,0.6883,Carlitos Junior,0.6257


**Para concluir:** ¿quién encabeza cada periodo? ¿Los primeros lugares por eigenvector coinciden con grado? Los componentes comparados tienen 191 y 545 artistas, respectivamente.


## 6. Homofilia: conexiones y atributos (56–68 min)

**Pregunta:** ¿hay más conexiones entre artistas semejantes respecto de un atributo? Primero usamos **país** y luego **generación**. Aquí país agrupa **Chile / Extranjero**; dos extranjeros cuentan como igual categoría aunque sean de países distintos.


### De dónde viene generación
No se calcula con el número de vínculos ni con la edad. **Importamos la columna `generation` de los archivos enriquecidos de mu-ch y la llamamos `generacion` en el Excel docente.** En la tesis se define a partir del año de inicio musical profesional. Si ese año no pudo establecerse, se utilizó como aproximación el año de la pista más antigua disponible en Spotify.

| Año de inicio | Categoría prevista |
|---|---|
| 1960–1969, 1970–1979, 1980–1989, 1990–1999 | 1960s, 1970s, 1980s, 1990s |
| 2000–2004 | 2000–2004 |
| 2005–2009 | 2005–2009 |
| 2010–2014 | 2010–2014 |
| 2015–2019 | 2015–2019 |
| 2020–2024 | 2020–2024 |

Por ejemplo, un inicio en 2003 corresponde a 2000–2004 y uno en 2017 a 2015–2019. Son ejemplos del criterio, no estimaciones nuevas. La generación viene ya categorizada: `factor()` y `as.integer()` solo la traducen a etiquetas numéricas para la función, **no vuelven a calcular el inicio**.

**Dos casos por revisar en la fuente:** Daimi tiene `year = 2025` y Siggy `year = 2026`, pero ambos figuran como 2020–2024 en P2. Esos años también exceden el final de observación. Conservamos las categorías suministradas para no corregirlas por conjetura. La comparación generacional reproduce ese dato pendiente de revisión, no valida su exactitud histórica.

### A. Aplicar asortatividad nominal
`assortativity_nominal()` resume la asociación entre categorías en los extremos de los vínculos. `factor()` reconoce las categorías y `as.integer()` les asigna códigos para entregarlos a la función; esos códigos no son un orden de países o generaciones.

Positivo indica asociación entre categorías iguales; negativo, entre distintas. Cerca de cero indica poca asociación bajo su referencia de mezcla, que considera la participación de las categorías en los extremos de los vínculos. No es una prueba de preferencia consciente ni de significación estadística.


In [ ]:
# Convertimos las categorías de país a códigos.
categorias_pais1 <- as.integer(factor(V(red1)$pais))
# Calculamos la asociación por país en P1.
asortatividad_pais1 <- assortativity_nominal(red1, categorias_pais1, directed = FALSE)
# Convertimos las generaciones a códigos.
categorias_generacion1 <- as.integer(factor(V(red1)$generacion))
# Calculamos la asociación por generación en P1.
asortatividad_generacion1 <- assortativity_nominal(red1, categorias_generacion1, directed = FALSE)
# Mostramos ambos resultados.
asortatividad_pais1
asortatividad_generacion1


### Ejercicio 4 · aplicar la función a P2 (2 min)
El docente calculó país y generación para P1. Copia el ejemplo y cambia **red1 por red2**, guardando las categorías y resultados con sufijo **2**. Aplica `assortativity_nominal()` a los dos atributos. ¿El signo se mantiene en el segundo periodo?


In [ ]:
# Tu turno: aplica el ejemplo docente a P2.
# Copia las cuatro líneas de cálculo, cambia red1 por red2 y usa nombres con sufijo 2.
# Al terminar, muestra los dos resultados.


### Comparar después del ejercicio 4
Comprobamos P2 con las mismas dos funciones de cálculo aplicadas por el docente y armamos una tabla. Aquí **cada fila representa el mismo atributo en ambos periodos**.


In [ ]:
# Convertimos país a códigos y calculamos su asortatividad en P2.
categorias_pais2 <- as.integer(factor(V(red2)$pais))
# Calculamos la asociación entre las categorías de los artistas conectados.
asortatividad_pais2 <- assortativity_nominal(red2, categorias_pais2, directed = FALSE)
# Repetimos para generación en P2.
categorias_generacion2 <- as.integer(factor(V(red2)$generacion))
# Calculamos la asociación entre las categorías de los artistas conectados.
asortatividad_generacion2 <- assortativity_nominal(red2, categorias_generacion2, directed = FALSE)
# Juntamos los resultados sin mezclar atributos entre periodos.
comparacion_homofilia <- data.frame(
  atributo = c("Pais: Chile / Extranjero", "Generacion"),
  P1 = round(c(asortatividad_pais1, asortatividad_generacion1), 4),
  P2 = round(c(asortatividad_pais2, asortatividad_generacion2), 4))
# Mostramos la tabla.
comparacion_homofilia


atributo,P1,P2
Pais: Chile / Extranjero,-0.0832,-0.136
Generacion,0.1011,0.101


### Cómo interpretar estos resultados concretos
- **País:** P1 da aproximadamente **−0,083** y P2 **−0,136**. Bajo la referencia de mezcla del índice, hay una asociación negativa entre categorías iguales. No significa que la mayoría de vínculos sea internacional: los vínculos de igual categoría son aproximadamente **84,6 % y 76,1 %**. Con una categoría chilena mayoritaria y con mucha participación en vínculos, ese porcentaje puede ser inferior al esperado por la referencia del índice.
- **Generación:** ambos periodos dan aproximadamente **+0,101**. Es una asociación positiva modesta entre categorías iguales, muy semejante en las dos ventanas. **0,101 no significa que el 10,1 % de los vínculos sea entre la misma generación**; es un coeficiente que compara mezcla observada y de referencia.
- El tamaño de los grupos y las restricciones de frontera importan. Estos coeficientes describen los datos suministrados; no establecen por sí solos selección consciente, influencia ni significación estadística. La variable de generación conserva los dos casos inconsistentes señalados antes.


**Para concluir:** ¿cada valor es positivo o negativo? ¿Se mantiene la asociación entre categorías iguales o distintas? La frontera de mu-ch restringe pares externos–externos: país no permite inferir preferencias individuales por sí solo.


### B. Ver quién se conecta y considerar las parejas posibles
La asortatividad es un resumen; conviene acompañarla de conteos. `ends()` devuelve los extremos de cada vínculo. Con `names = FALSE` obtenemos posiciones que permiten recuperar el país de cada extremo. La comparación `==` produce TRUE si coinciden y FALSE si no. `mean()` calcula su proporción; multiplicar por 100 la expresa como porcentaje.


In [ ]:
# Identificamos los dos extremos de cada vínculo.
extremos1 <- ends(red1, E(red1), names = FALSE)
# Recuperamos el país de uno de los extremos del vínculo.
pais_a1 <- V(red1)$pais[extremos1[, 1]]
# Recuperamos el país de uno de los extremos del vínculo.
pais_b1 <- V(red1)$pais[extremos1[, 2]]
# Calculamos el porcentaje de vínculos de igual categoría.
100 * mean(pais_a1 == pais_b1)


Un porcentaje alto de vínculos de igual categoría puede deberse a que hay muchos artistas de esa categoría. Por eso miramos **densidad dentro y entre categorías**: qué parte de las parejas posibles se conecta.

`induced_subgraph()` crea la subred de artistas que cumplen la condición; `edge_density()` calcula su densidad interna. Para relaciones entre Chile y Extranjero, dividimos sus vínculos observados por todas las parejas posibles entre los dos conjuntos. El símbolo `!=` significa “distinto”. Este bloque es una demostración guiada; no hace falta memorizar el cálculo del denominador.


In [ ]:
# Conservamos los nodos elegidos y sus vínculos entre sí.
chile1 <- induced_subgraph(red1, V(red1)[pais == "Chile"])
# Conservamos los nodos elegidos y sus vínculos entre sí.
exterior1 <- induced_subgraph(red1, V(red1)[pais == "Extranjero"])

# Calculamos la densidad dentro del grupo elegido.
densidad_chile1 <- edge_density(chile1, loops = FALSE)
# Calculamos la densidad dentro del grupo elegido.
densidad_exterior1 <- edge_density(exterior1, loops = FALSE)
# Contamos los vínculos entre categorías distintas.
vinculos_cruzados1 <- sum(pais_a1 != pais_b1)
# Contamos todas las parejas posibles entre los dos grupos.
parejas_cruzadas1 <- vcount(chile1) * vcount(exterior1)
# Dividimos vínculos observados por parejas posibles.
densidad_cruzada1 <- vinculos_cruzados1 / parejas_cruzadas1

# Juntamos los resultados en una tabla.
data.frame(tipo_de_pareja = c("Chile-Chile", "Extranjero-Extranjero", "Chile-Extranjero"),
           densidad = c(densidad_chile1, densidad_exterior1, densidad_cruzada1))


### Repetición guiada: densidades por país en P2
Repite la demostración anterior cambiando red1 por red2 y los sufijos 1 por 2. La comprobación agrega además el porcentaje de vínculos de igual categoría. Las tres densidades son proporciones; la última fila es un porcentaje.


In [ ]:
# Tu turno: copia aquí el cálculo que el docente hizo para P1.
# Cambia la red a red2 y guarda los resultados con sufijo 2.
# Después ejecuta la tabla de comprobación de la siguiente celda.


In [ ]:
# Recuperamos los países de los dos extremos de cada vínculo en P2.
extremos2 <- ends(red2, E(red2), names = FALSE)
# Recuperamos el país de uno de los extremos del vínculo.
pais_a2 <- V(red2)$pais[extremos2[, 1]]
# Recuperamos el país de uno de los extremos del vínculo.
pais_b2 <- V(red2)$pais[extremos2[, 2]]
# Seleccionamos los dos grupos del segundo periodo.
chile2 <- induced_subgraph(red2, V(red2)[pais == "Chile"])
# Conservamos los nodos elegidos y sus vínculos entre sí.
exterior2 <- induced_subgraph(red2, V(red2)[pais == "Extranjero"])
# Dividimos vínculos cruzados por parejas cruzadas posibles.
densidad_cruzada2 <- sum(pais_a2 != pais_b2) / (vcount(chile2) * vcount(exterior2))
# Construimos una fila por resultado y una columna por periodo.
comparacion_densidades <- data.frame(
  medida = c("Densidad Chile-Chile", "Densidad Extranjero-Extranjero",
             "Densidad Chile-Extranjero", "Vinculos misma categoria (%)"),
  P1 = round(c(densidad_chile1, densidad_exterior1, densidad_cruzada1,
               100 * mean(pais_a1 == pais_b1)), 4),
  P2 = round(c(edge_density(chile2, loops = FALSE), edge_density(exterior2, loops = FALSE),
               densidad_cruzada2, 100 * mean(pais_a2 == pais_b2)), 4))
# Mostramos la tabla comparativa.
comparacion_densidades


medida,P1,P2
Densidad Chile-Chile,0.0247,0.0202
Densidad Extranjero-Extranjero,0,0
Densidad Chile-Extranjero,0.0083,0.005
Vinculos misma categoria (%),84.6341,76.0608


**Para concluir:** ¿el porcentaje de vínculos de igual categoría cuenta la misma historia que las densidades? Recuerda que el cero externo–externo está condicionado por la frontera de selección de mu-ch.


**Límite del diseño de mu-ch:** sus vínculos requieren un extremo del núcleo validado. Esto restringe ciertas relaciones externas–externas. Un cero en ese bloque puede ser consecuencia del filtro; no demuestra rechazo entre artistas extranjeros. Los resultados son descriptivos y están condicionados por la cobertura de la investigación.

## 7. Aplicar Louvain y Leiden (68–78 min)

Ahora agrupamos por patrones de conexiones, no por un atributo dado. **Louvain** busca comunidades optimizando modularidad; **Leiden** añade un refinamiento con garantías de conectividad interna bajo sus condiciones de aplicación.

`set.seed(123)` fija el punto de partida aleatorio para repetir el ejemplo en el mismo entorno. Aplicamos las funciones directamente. En ambos casos ignoramos pesos y usamos modularidad con resolución 1. En Leiden elegimos ese criterio explícitamente porque también admite CPM.


In [ ]:
# Fijamos el azar para poder repetir el ejemplo.
set.seed(123)
# Buscamos comunidades con Louvain.
louvain1 <- cluster_louvain(red1, weights = NA, resolution = 1)

# Fijamos el azar para poder repetir el ejemplo.
set.seed(123)
# Buscamos comunidades con Leiden.
leiden1 <- cluster_leiden(red1, objective_function = "modularity",
                          weights = NA, resolution = 1, n_iterations = 10)


### Leer lo que devuelven las funciones
`membership()` devuelve la comunidad asignada a cada nodo. `table()` cuenta cuántos nodos hay en cada una. `groups()` recupera los grupos y `length()` cuenta cuántos son. `modularity()` mide el agrupamiento respecto de una referencia de conexiones esperadas según grado; no es un porcentaje de personas correctamente clasificadas.


In [ ]:
# Contamos cuántos artistas tiene cada comunidad.
table(membership(louvain1))
# Contamos cuántos artistas tiene cada comunidad.
table(membership(leiden1))
# Contamos cuántas comunidades hay.
length(groups(louvain1))
# Contamos cuántas comunidades hay.
length(groups(leiden1))
# Calculamos la modularidad del agrupamiento.
modularity(red1, membership(louvain1), weights = NA, resolution = 1)
# Calculamos la modularidad del agrupamiento.
modularity(red1, membership(leiden1), weights = NA, resolution = 1)


In [ ]:
# Dibujamos este grafo para observar su estructura.
ver_red(red1, "P1: Louvain", louvain1)


**Lectura preliminar:** esta ejecución de Louvain propone **16 comunidades** en P1. Los colores representan agrupamientos de conexiones, no países ni géneros musicales. Observa los sectores de la red que reciben colores distintos. Una comunidad tampoco es lo mismo que un componente: puede mantener vínculos con otras comunidades. Son resultados binarios de clase; no reproducen directamente las particiones ponderadas y seleccionadas de la tesis.


In [ ]:
# Dibujamos este grafo para observar su estructura.
ver_red(red1, "P1: Leiden", leiden1)


**Lectura preliminar:** esta ejecución de Leiden propone **17 comunidades** en P1. Los colores representan agrupamientos de conexiones, no países ni géneros musicales. Observa los sectores de la red que reciben colores distintos. Una comunidad tampoco es lo mismo que un componente: puede mantener vínculos con otras comunidades. Son resultados binarios de clase; no reproducen directamente las particiones ponderadas y seleccionadas de la tesis.


### Ejercicio 5 · aplicar ambos algoritmos a P2 (3 min)
Copia las llamadas resueltas a `cluster_louvain()` y `cluster_leiden()`. Cambia **red1 por red2**, guarda como **louvain2** y **leiden2**, y repite `table(membership(...))`. ¿Obtienen el mismo número de comunidades? Como extensión, dibuja ambos resultados con `ver_red()`.

Una comunidad detectada no equivale automáticamente a un género musical o identidad. Los colores y números son etiquetas; no identifican el mismo grupo entre algoritmos ni entre periodos. Una sola ejecución no demuestra que exista una única partición verdadera.


In [ ]:
# Copia aquí las llamadas a cluster_louvain() y cluster_leiden() para red2.
# Después aplica table(membership(...)) a los dos resultados.


### Comparar después del ejercicio 5
Ejecuta la comprobación tras intentar ambos algoritmos en P2. Comparamos cada método consigo mismo entre periodos. Los números y colores de comunidad no identifican automáticamente al mismo grupo.


In [ ]:
# Aplicamos Louvain al segundo periodo para comprobar el ejercicio.
set.seed(123)
# Buscamos comunidades con Louvain.
louvain2 <- cluster_louvain(red2, weights = NA, resolution = 1)
# Aplicamos Leiden con el mismo criterio: modularidad y resolución 1.
set.seed(123)
# Buscamos comunidades con Leiden.
leiden2 <- cluster_leiden(red2, objective_function = "modularity",
                          weights = NA, resolution = 1, n_iterations = 10)
# Una fila por método; mostramos cantidad de grupos y modularidad.
comparacion_comunidades <- data.frame(
  metodo = c("Louvain", "Leiden"),
  comunidades_P1 = c(length(groups(louvain1)), length(groups(leiden1))),
  comunidades_P2 = c(length(groups(louvain2)), length(groups(leiden2))),
  modularidad_P1 = round(c(modularity(red1, membership(louvain1), weights = NA),
                          modularity(red1, membership(leiden1), weights = NA)), 4),
  modularidad_P2 = round(c(modularity(red2, membership(louvain2), weights = NA),
                          modularity(red2, membership(leiden2), weights = NA)), 4))
# Mostramos la comparación.
comparacion_comunidades


metodo,comunidades_P1,comunidades_P2,modularidad_P1,modularidad_P2
Louvain,16,19,0.6976,0.573
Leiden,17,19,0.7025,0.5845


**Para concluir:** ¿cada método encuentra más o menos grupos en P2? ¿Ambos métodos coinciden en el número? Con redes de tamaños y composiciones diferentes, una diferencia de modularidad no basta para explicar un cambio histórico ni para nombrar comunidades como géneros musicales.


## 8. Cierre: leer una tabla y explicar lo observado (78–80 min)
Elige **una de las tablas comparativas que ya calculamos**. No tienes que ejecutar código nuevo. Explica en tus palabras qué cambia entre P1 y P2 y qué conclusión permiten esos datos.

Puedes usar uno de estos ejemplos como guía:

1. **Red completa:** “En P2 hay más artistas y colaboraciones, pero una menor proporción de todas las parejas posibles colabora”. Explica el denominador de densidad.
2. **Centralidad:** identifica un artista en los rankings y explica qué significa su grado o su posición de puente. Evita convertir centralidad en una medida de talento.
3. **Homofilia:** lee el signo de la asortatividad por generación y explica qué atributo se comparó. Considera los dos casos de inicio pendientes de revisión.
4. **Comunidades:** compara los grupos propuestos por Louvain y Leiden, sin asumir que cada grupo es un género musical.

La pregunta final es: **¿qué información histórica o cualitativa necesitaríamos para explicar ese cambio?** Las soluciones se repiten a continuación para revisar las instrucciones después de la clase.


## Soluciones y extensiones · consultar después de intentar
Estos cálculos aparecen al final, no se ejecutan en la preparación inicial. Repetimos el procedimiento con P2. Los bloques se pueden ejecutar completos aunque el ejercicio haya quedado sin terminar.


In [ ]:
# Ejercicio 1: medidas de red completa en P2.
vcount(red2)
# Contamos las parejas que colaboran.
ecount(red2)
# Calculamos la proporción de parejas posibles que colaboran.
edge_density(red2, loops = FALSE)
# Calculamos el promedio de colaboradores por artista.
mean(degree(red2))

# Extensión: componentes, aislados y centralización.
componentes2 <- components(red2)
# Mostramos la cantidad de componentes.
componentes2$no
# Mostramos cuántos artistas tiene el componente mayor.
max(componentes2$csize)
# Calculamos qué porcentaje de artistas está en el componente mayor.
100 * max(componentes2$csize) / vcount(red2)
# Contamos los artistas que no tienen colaboradores.
sum(degree(red2) == 0)
# Calculamos cuánto se concentran los contactos en la red.
centr_degree(red2, loops = FALSE, normalized = TRUE)$centralization


In [ ]:
# Ejercicio 2: calcular y ordenar grado.
grado2 <- degree(red2)
# Juntamos los resultados en una tabla.
centralidad2 <- data.frame(artista = V(red2)$artista, grado = as.numeric(grado2))
# Ordenamos las filas por la medida, de mayor a menor.
ranking_grado2 <- centralidad2[order(centralidad2$grado, decreasing = TRUE), ]
rownames(ranking_grado2) <- NULL  # Vuelve a numerar las filas para facilitar la lectura
# Mostramos las primeras filas de la tabla.
head(ranking_grado2, 10)

# Extensión: calcular y ordenar intermediación.
centralidad2$intermediacion <- betweenness(red2, directed = FALSE,
                                         weights = NA, normalized = TRUE)
# Ordenamos las filas por la medida, de mayor a menor.
ranking_puentes2 <- centralidad2[order(centralidad2$intermediacion, decreasing = TRUE), ]
rownames(ranking_puentes2) <- NULL  # Vuelve a numerar las filas para facilitar la lectura
# Mostramos las primeras filas de la tabla.
head(ranking_puentes2, 10)


In [ ]:
# Ejercicio 3: seleccionar componente, calcular y ordenar eigenvector.
componentes2 <- components(red2)
# Buscamos cuál es el componente más grande.
mayor2 <- which.max(componentes2$csize)
# Conservamos los nodos elegidos y sus vínculos entre sí.
principal2 <- induced_subgraph(red2, which(componentes2$membership == mayor2))
# Calculamos eigenvector dentro del componente mayor.
eigen2 <- eigen_centrality(principal2, directed = FALSE, weights = NA)$vector
# Juntamos los resultados en una tabla.
tabla_eigen2 <- data.frame(artista = V(principal2)$artista, eigenvector = as.numeric(eigen2))
# Ordenamos las filas por la medida, de mayor a menor.
ranking_eigen2 <- tabla_eigen2[order(tabla_eigen2$eigenvector, decreasing = TRUE), ]
rownames(ranking_eigen2) <- NULL  # Vuelve a numerar las filas para facilitar la lectura
# Mostramos las primeras filas de la tabla.
head(ranking_eigen2, 10)


In [ ]:
# Ejercicio 4: misma función, otro atributo.
generaciones1 <- as.integer(factor(V(red1)$generacion))
# Calculamos la asociación entre las categorías de los artistas conectados.
assortativity_nominal(red1, generaciones1, directed = FALSE)

# Extensión: mismo atributo, otro periodo.
generaciones2 <- as.integer(factor(V(red2)$generacion))
# Calculamos la asociación entre las categorías de los artistas conectados.
assortativity_nominal(red2, generaciones2, directed = FALSE)


In [ ]:
# Ejercicio 5: aplicar los algoritmos, sin resultados precomputados.
set.seed(123)
# Buscamos comunidades con Louvain.
louvain2 <- cluster_louvain(red2, weights = NA, resolution = 1)
# Fijamos el azar para poder repetir el ejemplo.
set.seed(123)
# Buscamos comunidades con Leiden.
leiden2 <- cluster_leiden(red2, objective_function = "modularity",
                          weights = NA, resolution = 1, n_iterations = 10)
# Contamos cuántos artistas tiene cada comunidad.
table(membership(louvain2))
# Contamos cuántos artistas tiene cada comunidad.
table(membership(leiden2))
# Contamos cuántas comunidades hay.
length(groups(louvain2))
# Contamos cuántas comunidades hay.
length(groups(leiden2))
# Calculamos la modularidad del agrupamiento.
modularity(red2, membership(louvain2), weights = NA, resolution = 1)
# Calculamos la modularidad del agrupamiento.
modularity(red2, membership(leiden2), weights = NA, resolution = 1)
# Dibujamos la red indicada.


In [ ]:
# Dibujamos este grafo para observar su estructura.
ver_red(red2, "P2: Louvain", louvain2)


**Lectura preliminar:** esta ejecución de Louvain propone **19 comunidades** en P2. Los colores representan agrupamientos de conexiones, no países ni géneros musicales. Observa los sectores de la red que reciben colores distintos. Una comunidad tampoco es lo mismo que un componente: puede mantener vínculos con otras comunidades. Son resultados binarios de clase; no reproducen directamente las particiones ponderadas y seleccionadas de la tesis.


In [ ]:
# Dibujamos este grafo para observar su estructura.
ver_red(red2, "P2: Leiden", leiden2)


**Lectura preliminar:** esta ejecución de Leiden propone **19 comunidades** en P2. Los colores representan agrupamientos de conexiones, no países ni géneros musicales. Observa los sectores de la red que reciben colores distintos. Una comunidad tampoco es lo mismo que un componente: puede mantener vínculos con otras comunidades. Son resultados binarios de clase; no reproducen directamente las particiones ponderadas y seleccionadas de la tesis.


## Guardar resultados · opcional después de las soluciones
`write.csv()` guarda una tabla. Podrás descargarla desde Archivos. `sessionInfo()` registra las versiones del entorno utilizado.


In [ ]:
# Guardamos la tabla para poder descargarla.
write.csv(centralidad2, "centralidad_P2.csv", row.names = FALSE, fileEncoding = "UTF-8")
# Guardamos la tabla para poder descargarla.
write.csv(ranking_eigen2, "ranking_eigenvector_P2.csv", row.names = FALSE, fileEncoding = "UTF-8")
# Mostramos las versiones de R y sus paquetes.
sessionInfo()


## Fuentes y decisiones
- **Datos reales:** archivos finales `nodes_*`, `edges_*` y `artist_attributes_*_enriched.csv` de `mu-ch/data/final`. Se conservan IDs y atributos por periodo. Pares con al menos una pista se convierten en vínculos binarios, sin pesos ni lazos. P1: 213 artistas y 410 parejas. P2: 564 artistas y 1.579 parejas. Las ventanas originales son independientes y de distinta duración.
- **Diseño empírico:** Muñoz Tapia, S. M. (2026). *Networks, Categories, and Collaboration: The Emergence and Consolidation of Chilean Música Urbana*. MSc Social Network Analysis, University of Manchester. Se conserva la frontera del núcleo validado y sus colaboradores; ciertas ausencias derivan de esa cobertura.
- **Conceptos:** Crossley, N., McAndrew, S. y Widdop, P. (2015). *What is social network analysis? An introduction for music scholars*. En *Social Networks and Music Worlds*, capítulo 2. Se distingue formato de datos de alcance, modos y dirección de la red. Leiden es posterior al libro.
- **Funciones específicas:** [crear grafos desde tablas](https://r.igraph.org/reference/graph_from_data_frame.html), [grado](https://r.igraph.org/reference/degree.html), [intermediación](https://r.igraph.org/reference/betweenness.html), [eigenvector](https://r.igraph.org/reference/eigen_centrality.html), [asortatividad](https://r.igraph.org/reference/assortativity.html), [Louvain](https://r.igraph.org/reference/cluster_louvain.html), [Leiden](https://r.igraph.org/reference/cluster_leiden.html).
- **Paquetes:** readxl para Excel; igraph para análisis; R base para tablas, conteos y ordenamiento; ggraph/ggplot2 para dibujos. Solo `ver_red()` y `ver_ego()` son funciones auxiliares docentes: dibujan sin calcular rankings, tablas de homofilia ni particiones de comunidades por adelantado.
- **Ejemplo egocentrado:** todos los vínculos y contactos son ficticios. El nombre de ego alude a Pablo Chill-E, pero no afirmamos relaciones laborales reales. Los Excel real y ficticio permanecen separados.
- **Colab:** [versiones de entorno](https://research.google.com/colaboratory/runtime-version-faq.html). Guarda el cuaderno en Drive. La semilla permite repetir las ejecuciones estocásticas en un mismo entorno; resultados pueden cambiar con versiones diferentes.
